In [ ]:
import cv2
import numpy as np

## TO STACK ALL THE IMAGES IN ONE WINDOW
def stackImages(imgArray,scale,lables=[]):
    rows = len(imgArray)
    cols = len(imgArray[0])
    rowsAvailable = isinstance(imgArray[0], list)
    width = imgArray[0][0].shape[1]
    height = imgArray[0][0].shape[0]
    if rowsAvailable:
        for x in range ( 0, rows):
            for y in range(0, cols):
                imgArray[x][y] = cv2.resize(imgArray[x][y], (0, 0), None, scale, scale)
                if len(imgArray[x][y].shape) == 2: imgArray[x][y]= cv2.cvtColor( imgArray[x][y], cv2.COLOR_GRAY2BGR)
        imageBlank = np.zeros((height, width, 3), np.uint8)
        hor = [imageBlank]*rows
        hor_con = [imageBlank]*rows
        for x in range(0, rows):
            hor[x] = np.hstack(imgArray[x])
            hor_con[x] = np.concatenate(imgArray[x])
        ver = np.vstack(hor)
        ver_con = np.concatenate(hor)
    else:
        for x in range(0, rows):
            imgArray[x] = cv2.resize(imgArray[x], (0, 0), None, scale, scale)
            if len(imgArray[x].shape) == 2: imgArray[x] = cv2.cvtColor(imgArray[x], cv2.COLOR_GRAY2BGR)
        hor= np.hstack(imgArray)
        hor_con= np.concatenate(imgArray)
        ver = hor
    if len(lables) != 0:
        eachImgWidth= int(ver.shape[1] / cols)
        eachImgHeight = int(ver.shape[0] / rows)
        #print(eachImgHeight)
        for d in range(0, rows):
            for c in range (0,cols):
                cv2.rectangle(ver,(c*eachImgWidth,eachImgHeight*d),(c*eachImgWidth+len(lables[d][c])*13+27,30+eachImgHeight*d),(255,255,255),cv2.FILLED)
                cv2.putText(ver,lables[d][c],(eachImgWidth*c+10,eachImgHeight*d+20),cv2.FONT_HERSHEY_COMPLEX,0.7,(255,0,255),2)
    return ver

def reorder(myPoints):

    myPoints = myPoints.reshape((4, 2)) # REMOVE EXTRA BRACKET
    print(myPoints)
    myPointsNew = np.zeros((4, 1, 2), np.int32) # NEW MATRIX WITH ARRANGED POINTS
    add = myPoints.sum(1)
    print(add)
    print(np.argmax(add))
    myPointsNew[0] = myPoints[np.argmin(add)]  #[0,0]
    myPointsNew[3] =myPoints[np.argmax(add)]   #[w,h]
    diff = np.diff(myPoints, axis=1)
    myPointsNew[1] =myPoints[np.argmin(diff)]  #[w,0]
    myPointsNew[2] = myPoints[np.argmax(diff)] #[h,0]

    return myPointsNew

def rectContour(contours):

    rectCon = []
    max_area = 0
    for i in contours:
        area = cv2.contourArea(i)
        if area > 50:
            peri = cv2.arcLength(i, True)
            approx = cv2.approxPolyDP(i, 0.02 * peri, True)
            if len(approx) == 4:
                rectCon.append(i)
    rectCon = sorted(rectCon, key=cv2.contourArea,reverse=True)
    #print(len(rectCon))
    return rectCon

def getCornerPoints(cont):
    peri = cv2.arcLength(cont, True) # LENGTH OF CONTOUR
    approx = cv2.approxPolyDP(cont, 0.02 * peri, True) # APPROXIMATE THE POLY TO GET CORNER POINTS
    return approx

def preprocess_image(imgResized, blur_ksize=(5,5), blur_sigma=1, canny_thresh1=10, canny_thresh2=50):
    results = {}
    results["gray"] = cv2.cvtColor(imgResized, cv2.COLOR_BGR2GRAY)
    results["blur"] = cv2.GaussianBlur(results["gray"], blur_ksize, blur_sigma)
    results["canny"] = cv2.Canny(results["blur"], canny_thresh1, canny_thresh2)
    results["blank"] = np.zeros_like(imgResized)

    return results

def trim_margins(bgr):
    # 1) build a mask of non-white pixels (red rings + black fills)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    # red/orange ranges (two lobes in HSV) + dark ink
    lower_red1 = np.array([0, 80, 80]);   upper_red1 = np.array([12, 255, 255])
    lower_red2 = np.array([170, 80, 80]); upper_red2 = np.array([180,255, 255])
    mask_red = cv2.inRange(hsv, lower_red1, upper_red1) | cv2.inRange(hsv, lower_red2, upper_red2)

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    mask_dark = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    mask = cv2.bitwise_or(mask_red, mask_dark)

    # clean small specks and connect nearby edges
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8), iterations=1)

    # 2) find tight bounding box around non-zero pixels
    nz = cv2.findNonZero(mask)
    if nz is None:
        return bgr  # nothing found; return input unchanged

    x, y, w, h = cv2.boundingRect(nz)
    trimmed = bgr[y:y+h, x:x+w]

    return trimmed

def get_width_height(points):
    # width = max distance between left/right pairs
    w1 = np.linalg.norm(points[0][0] - points[1][0])  # top-left ↔ top-right
    w2 = np.linalg.norm(points[2][0] - points[3][0])  # bottom-left ↔ bottom-right
    width = int(max(w1, w2))

    # height = max distance between top/bottom pairs
    h1 = np.linalg.norm(points[0][0] - points[2][0])  # top-left ↔ bottom-left
    h2 = np.linalg.norm(points[1][0] - points[3][0])  # top-right ↔ bottom-right
    height = int(max(h1, h2))

    return width, height

def grayscale_threshold(img, threshold_value=170, invert=True):
  # Convert to grayscale
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  # Apply thresholding
  if invert:
    thresh = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY_INV)[1]
  else: thresh = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY)[1]

  return {
      'gray': gray,
      'thresh': thresh
    }

def split_number_into_bubbles(image, num_rows=10, num_cols=6):
    # Split into rows (numbers: 0-9)
    rows = np.array_split(image, num_rows, axis=0)

    bubbles = []

    for row_index, row in enumerate(rows):
        # Split each row into columns (positions)
        columns = np.array_split(row, num_cols, axis=1)
        for col_index, bubble_img in enumerate(columns):  # Fixed variable name
            # Determine number based on row (row 0 = number 0, row 1 = number 1, etc.)
            number = row_index  # This gives 0, 1, 2, ..., 9

            bubbles.append({
                'image': bubble_img,  # Fixed: use bubble_img, not bubble
                'number': number,
                'position': col_index + 1,
                'index': len(bubbles)
            })
    return bubbles


## NAME SECTION
def split_name_into_bubbles(image, num_rows=27, num_cols=30):
    # Split into rows (letters: SPACE, A-Z)
    rows = np.array_split(image, num_rows, axis=0)

    bubbles = []

    for row_index, row in enumerate(rows):
        # Split each row into columns (positions in name)
        columns = np.array_split(row, num_cols, axis=1)
        for col_index, bubble in enumerate(columns):
            # Determine letter based on row
            if row_index == 0:
                letter = 'SPACE'  # First row is space
            else:
                letter = chr(64 + row_index)  # A-Z (row1=SPACE, row2=A, row3=B, etc.)

            bubbles.append({
                'image': bubble,
                'letter': letter,           # SPACE, A, B, C, ..., Z
                'position': col_index + 1,  # Position 1-27 in name
                'index': len(bubbles)
            })
    return bubbles

def process_name_column(name_image, num_positions=30, num_letters=27, threshold=1000):
    all_bubbles_data = []
    name_matrix = np.zeros((num_letters, num_positions), dtype=int)  # 27 letters × 30 positions

    # Apply grayscale and threshold
    name_res = grayscale_threshold(name_image)
    name_thresh = name_res['thresh']

    # Split into bubbles
    bubbles = split_name_into_bubbles(name_thresh, num_rows=num_letters, num_cols=num_positions)

    # Process each bubble
    for bubble in bubbles:
        # Convert letter to row index
        if bubble['letter'] == 'SPACE':
            letter_index = 0
        else:
            letter_index = ord(bubble['letter']) - 64  # A=1, B=2, ..., Z=26

        # Position is 1-30, convert to 0-29 index
        position_index = bubble['position'] - 1

        # Count non-zero pixels
        pixel_value = cv2.countNonZero(bubble['image'])

        # Store in matrix (letters × positions)
        name_matrix[letter_index, position_index] = pixel_value

        # Store detailed information
        all_bubbles_data.append({
            'position': bubble['position'],
            'letter': bubble['letter'],
            'pixel_value': pixel_value,
            'image': bubble['image']
        })

    return name_matrix, all_bubbles_data


## REGISTRANT NUMBER SECTION
def process_registrant_number_column(reg_image, num_cols=6, num_rows=10, threshold=1000):
    all_bubbles_data = []
    reg_matrix = np.zeros((num_rows, num_cols), dtype=int)  # 10 numbers × 6 positions

    # Apply grayscale and threshold
    reg_res = grayscale_threshold(reg_image)
    reg_thresh = reg_res['thresh']

    # Split into bubbles
    bubbles = split_number_into_bubbles(reg_thresh, num_rows=num_rows, num_cols=num_cols)

    # Process each bubble
    for bubble in bubbles:
        # Convert number to row index (0-9)
        if bubble['number'] == '0':
            number_index = 0
        else:
            number_index = int(bubble['number'])  # 1-9

        # Position is 1-6, convert to 0-5 index
        position_index = bubble['position'] - 1

        # Count non-zero pixels
        pixel_value = cv2.countNonZero(bubble['image'])

        # Store in matrix (numbers × positions)
        reg_matrix[number_index, position_index] = pixel_value  # Fixed: use reg_matrix

        # Store detailed information
        all_bubbles_data.append({
            'position': bubble['position'],
            'number': bubble['number'],
            'pixel_value': pixel_value,
            'image': bubble['image']
        })

    return reg_matrix, all_bubbles_data  # Fixed: return reg_matrix


## DATE SECTION
def process_date_number_column(reg_image, num_cols=6, num_rows=10, threshold=1000):
    all_bubbles_data = []
    date_matrix = np.zeros((num_rows, num_cols), dtype=int)  # 10 numbers × 6 positions

    # Apply grayscale and threshold
    date_res = grayscale_threshold(reg_image)
    date_thresh = date_res['thresh']

    # Split into bubbles
    bubbles = split_number_into_bubbles(date_thresh, num_rows=num_rows, num_cols=num_cols)

    # Process each bubble
    for bubble in bubbles:
        # Convert number to row index (0-9)
        if bubble['number'] == '0':
            number_index = 0
        else:
            number_index = int(bubble['number'])  # 1-9

        # Position is 1-6, convert to 0-5 index
        position_index = bubble['position'] - 1

        # Count non-zero pixels
        pixel_value = cv2.countNonZero(bubble['image'])

        # Store in matrix (numbers × positions)
        date_matrix[number_index, position_index] = pixel_value

        # Store detailed information
        all_bubbles_data.append({
            'position': bubble['position'],
            'number': bubble['number'],
            'pixel_value': pixel_value,
            'image': bubble['image']
        })

    # Remove specific indices with pixel value 0
    indices_to_remove = [12, 18, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58]
    filtered_bubbles_data = [bubble for i, bubble in enumerate(all_bubbles_data) if i not in indices_to_remove]

    return date_matrix, filtered_bubbles_data


## FORM SECTION
def split_form_into_bubbles(image, num_rows=10, num_cols=3):
    # Split into rows
    rows = np.array_split(image, num_rows, axis=0)

    bubbles = []

    for row_index, row in enumerate(rows):
        # Split each row into columns
        columns = np.array_split(row, num_cols, axis=1)
        for col_index, bubble_img in enumerate(columns):
            # Determine value based on column type
            if col_index < 2:  # First two columns are numbers (0-9)
                value = row_index  # This gives 0, 1, 2, ..., 9
                value_type = 'number'
            else:  # Third column is alphabet (A-J)
                value = chr(65 + row_index)  # This gives A, B, C, ..., J
                value_type = 'letter'

            bubbles.append({
                'image': bubble_img,
                'value': value,
                'type': value_type,
                'position': col_index + 1,
                'index': len(bubbles)
            })
    return bubbles

def process_form_number_column(image, num_cols=3, num_rows=10, threshold=1000):
    all_bubbles_data = []
    form_matrix = np.zeros((num_rows, num_cols), dtype=int)  # 10 rows × 3 columns

    # Apply grayscale and threshold
    form_res = grayscale_threshold(image)
    form_thresh = form_res['thresh']

    # Split into bubbles
    bubbles = split_form_into_bubbles(form_thresh, num_rows=num_rows, num_cols=num_cols)

    # Process each bubble
    for bubble in bubbles:
        # Get row index (0-9)
        row_index = bubble['value'] if bubble['type'] == 'number' else ord(bubble['value']) - 65

        # Position is 1-3, convert to 0-2 index
        position_index = bubble['position'] - 1

        # Count non-zero pixels
        pixel_value = cv2.countNonZero(bubble['image'])

        # Store in matrix
        form_matrix[row_index, position_index] = pixel_value

        # Store detailed information
        all_bubbles_data.append({
            'position': bubble['position'],
            'type': bubble['type'],
            'value': bubble['value'],
            'pixel_value': pixel_value,
            'image': bubble['image']
        })

    return form_matrix, all_bubbles_data


## BOOKLET SECTION
def split_booklet_into_bubbles(image, num_rows=10, num_cols=3):
    # Split into rows (numbers: 0-9)
    rows = np.array_split(image, num_rows, axis=0)

    bubbles = []

    for row_index, row in enumerate(rows):
        # Split each row into columns (positions)
        columns = np.array_split(row, num_cols, axis=1)
        for col_index, bubble_img in enumerate(columns):  # Fixed variable name
            # Determine number based on row (row 0 = number 0, row 1 = number 1, etc.)
            number = row_index  # This gives 0, 1, 2, ..., 9

            bubbles.append({
                'image': bubble_img,  # Fixed: use bubble_img, not bubble
                'number': number,
                'position': col_index + 1,
                'index': len(bubbles)
            })
    return bubbles

def process_booklet_number_column(image, num_cols=3, num_rows=10, threshold=1000):
    all_bubbles_data = []
    booklet_matrix = np.zeros((num_rows, num_cols), dtype=int)  # 10 numbers × 6 positions

    # Apply grayscale and threshold
    booklet_res = grayscale_threshold(image)
    booklet_thresh = booklet_res['thresh']

    # Split into bubbles
    bubbles = split_booklet_into_bubbles(booklet_thresh, num_rows=num_rows, num_cols=num_cols)

    # Process each bubble
    for bubble in bubbles:
        # Convert number to row index (0-9)
        if bubble['number'] == '0':
            number_index = 0
        else:
            number_index = int(bubble['number'])  # 1-9

        # Position is 1-6, convert to 0-5 index
        position_index = bubble['position'] - 1

        # Count non-zero pixels
        pixel_value = cv2.countNonZero(bubble['image'])

        # Store in matrix (numbers × positions)
        booklet_matrix[number_index, position_index] = pixel_value

        # Store detailed information
        all_bubbles_data.append({
            'position': bubble['position'],
            'number': bubble['number'],
            'pixel_value': pixel_value,
            'image': bubble['image']
        })

    return booklet_matrix, all_bubbles_data  # Fixed: return reg_matrix


## ANSWER SECTION
def split_answer_into_bubbles(image, num_rows=20, num_cols=4):
    # Split into rows (questions)
    rows = np.array_split(image, num_rows, axis=0)

    bubbles = []

    for row_index, row in enumerate(rows):
        # Split each row into columns (options)
        columns = np.array_split(row, num_cols, axis=1)

        for col_index, bubble in enumerate(columns):
            bubbles.append({
                'image': bubble,
                'question': row_index + 1,
                'option': chr(65 + col_index),  # A, B, C, D
                'index': len(bubbles)
            })

    return bubbles

def process_answer_columns(trimmed_cols, num_questions_per_col=20, num_options=4, threshold=1000):
    all_bubbles_data = []
    bubble_values_combined = np.zeros((100, 4), dtype=int)  # 100 questions × 4 options

    for col_index, col_img in enumerate(trimmed_cols):
        print(f"\n=== PROCESSING COLUMN {col_index + 1} ===")

        # Apply grayscale and threshold
        col_res = grayscale_threshold(col_img)
        col_thresh = col_res['thresh']

        # Split into bubbles
        bubbles = split_answer_into_bubbles(col_thresh, num_rows=num_questions_per_col, num_cols=num_options)

        # Calculate starting question number for this column
        start_question = col_index * num_questions_per_col + 1

        print(f"Column {col_index + 1} contains questions {start_question} to {start_question + num_questions_per_col - 1}")

        # Process each bubble in this column
        for bubble in bubbles:
            # Calculate global question number
            global_question = start_question + (bubble['question'] - 1)

            # Count non-zero pixels
            pixel_value = cv2.countNonZero(bubble['image'])

            # Store in the combined matrix
            row_index = global_question - 1  # 0-based index for matrix
            col_option = ord(bubble['option']) - 65  # A=0, B=1, C=2, D=3

            bubble_values_combined[row_index, col_option] = pixel_value

            # Also store detailed information
            all_bubbles_data.append({
                'global_question': global_question,
                'original_question': bubble['question'],
                'column': col_index + 1,
                'option': bubble['option'],
                'pixel_value': pixel_value,
                'image': bubble['image']
            })

    return bubble_values_combined, all_bubbles_data